In [1]:
# Understat xG Analysis for Attacking Players

"""Run the league stats script and visualise attacking players' xG and xGChain
per minute over the last 5 seasons, including a line of best fit.

This notebook assumes:
- `src/understat_league_stats.py` exists one level up from this notebook
- The script writes CSVs under `data/{year}/` (e.g. `data/2025/<league_code>_players_2025.csv`)
- You have `pandas`, `matplotlib`, `seaborn`, and `plotly` installed
  (see `requirements.txt`).
"""

from pathlib import Path

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

# Make plots a bit prettier by default
sns.set_theme(style="whitegrid")

PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data"  # league script writes CSVs under data/{season}/
SRC_DIR = PROJECT_ROOT / "src"


In [2]:
# Parameters

# Leagues to analyse – must match the league codes used
# in `src/understat_league_stats.py`.
LEAGUE_CODES = ["EPL", "La_liga", "Bundesliga", "Serie_A", "Ligue_1"]

# This should match `SEASON_START_YEAR` in `src/understat_league_stats.py`
SEASON_START_YEAR = 2025

# Number of seasons (including SEASON_START_YEAR) to include
N_SEASONS = 1

seasons = list(range(SEASON_START_YEAR, SEASON_START_YEAR - N_SEASONS, -1))
seasons, LEAGUE_CODES

([2025], ['EPL', 'La_liga', 'Bundesliga', 'Serie_A', 'Ligue_1'])

In [3]:
# Run the league stats script to (re)generate CSVs

import runpy

league_script_path = SRC_DIR / "understat_league_stats.py"
print(f"Running {league_script_path} ...")
runpy.run_path(str(league_script_path))
print("Done.")

Running /Users/markdavis/Projects/soccer-stats/src/understat_league_stats.py ...
Done.


In [4]:
# Load and combine player data for all selected leagues and seasons

player_dfs = []

for league_code in LEAGUE_CODES:
    for season in seasons:
        csv_path = DATA_DIR / str(season) / f"{league_code}_players_{season}.csv"
        print(f"Loading {csv_path} ...")
        df = pd.read_csv(csv_path)
        df["season"] = season
        df["league"] = league_code
        player_dfs.append(df)

players = pd.concat(player_dfs, ignore_index=True)
print(f"Combined players shape: {players.shape}")
players.head()

Loading /Users/markdavis/Projects/soccer-stats/data/2025/EPL_players_2025.csv ...
Loading /Users/markdavis/Projects/soccer-stats/data/2025/La_liga_players_2025.csv ...
Loading /Users/markdavis/Projects/soccer-stats/data/2025/Bundesliga_players_2025.csv ...
Loading /Users/markdavis/Projects/soccer-stats/data/2025/Serie_A_players_2025.csv ...
Loading /Users/markdavis/Projects/soccer-stats/data/2025/Ligue_1_players_2025.csv ...
Combined players shape: (2595, 20)


,id,player_name,games,time,goals,xG,assists,xA,shots,key_passes,yellow_cards,red_cards,position,team_title,npg,npxG,xGChain,xGBuildup,season,league
0,8260,Erling Haaland,26,2169,22,21.904284,6,4.323697,92,19,1,0,F S,Manchester City,19,18.859608,23.772826,3.301901,2025,EPL
1,13222,Thiago,26,2212,17,18.374264,1,2.371154,61,14,5,0,F S,Brentford,11,13.046082,16.014959,2.890031,2025,EPL
2,11363,Antoine Semenyo,25,2250,13,9.511308,4,2.991685,59,29,6,0,F M,"Bournemouth,Manchester City",12,7.988970,12.810340,3.261103,2025,EPL
3,5555,Dominic Calvert-Lewin,23,1671,10,10.760022,1,1.317244,45,13,1,0,F S,Leeds,8,9.237684,11.817670,1.887144,2025,EPL
4,8272,João Pedro,26,1913,10,9.348644,4,3.503022,45,24,3,0,F M S,Chelsea,10,9.348644,12.301924,3.428458,2025,EPL


In [5]:
# Load UEFA coefficients and add to players dataframe
import sys
sys.path.insert(0, str(SRC_DIR))
from uefa_coefficient_scraper import fetch_uefa_coefficients, get_coefficient_for_association, save_coefficients

coeff_path = DATA_DIR / str(SEASON_START_YEAR) / f"uefa_coefficients_{SEASON_START_YEAR}.csv"
if coeff_path.exists():
    uefa_df = pd.read_csv(coeff_path)
    print(f"Loaded UEFA coefficients from {coeff_path}")
else:
    uefa_df = fetch_uefa_coefficients(SEASON_START_YEAR)
    save_coefficients(uefa_df, SEASON_START_YEAR)

league_to_coeff = {
    league: get_coefficient_for_association(uefa_df, league)
    for league in LEAGUE_CODES
}
players["uefa_coefficient"] = players["league"].map(league_to_coeff)
print(f"Added uefa_coefficient column. Sample: {players[['league', 'uefa_coefficient']].drop_duplicates().to_string(index=False)}")
players.head()

Loaded UEFA coefficients from /Users/markdavis/Projects/soccer-stats/data/2025/uefa_coefficients_2025.csv
Added uefa_coefficient column. Sample:     league  uefa_coefficient
       EPL           111.797
   La_liga            90.734
Bundesliga            87.617
   Serie_A            96.446
   Ligue_1            79.212


,id,player_name,games,time,goals,xG,assists,xA,shots,key_passes,...,red_cards,position,team_title,npg,npxG,xGChain,xGBuildup,season,league,uefa_coefficient
0,8260,Erling Haaland,26,2169,22,21.904284,6,4.323697,92,19,...,0,F S,Manchester City,19,18.859608,23.772826,3.301901,2025,EPL,111.797
1,13222,Thiago,26,2212,17,18.374264,1,2.371154,61,14,...,0,F S,Brentford,11,13.046082,16.014959,2.890031,2025,EPL,111.797
2,11363,Antoine Semenyo,25,2250,13,9.511308,4,2.991685,59,29,...,0,F M,"Bournemouth,Manchester City",12,7.988970,12.810340,3.261103,2025,EPL,111.797
3,5555,Dominic Calvert-Lewin,23,1671,10,10.760022,1,1.317244,45,13,...,0,F S,Leeds,8,9.237684,11.817670,1.887144,2025,EPL,111.797
4,8272,João Pedro,26,1913,10,9.348644,4,3.503022,45,24,...,0,F M S,Chelsea,10,9.348644,12.301924,3.428458,2025,EPL,111.797


In [6]:
# Filter to attacking players

# The `position` column is a shorthand string like "F", "F S", "F M S", etc.
# We'll treat any player whose position string contains "F" (forward) as an
# attacking player.

attacking = players[players["position"].astype(str).str.contains("F")].copy()
print(f"Attacking players: {len(attacking)} of {len(players)} total rows")

# Compute per-minute rates (xG and xGChain per minute played)
# Guard against division by zero for players with 0 minutes.

attacking = attacking[attacking["time"] > 0].copy()
attacking["xG_per_min"] = attacking["xG"] / attacking["time"]
attacking["xGChain_per_min"] = attacking["xGChain"] / attacking["time"]
attacking = attacking[attacking["time"] > 1000]

attacking[["player_name", "team_title", "season", "time", "xG", "xG_per_min", "xGChain", "xGChain_per_min"]].head()

Attacking players: 605 of 2595 total rows


,player_name,team_title,season,time,xG,xG_per_min,xGChain,xGChain_per_min
0,Erling Haaland,Manchester City,2025,2169,21.904284,0.010099,23.772826,0.010960
1,Thiago,Brentford,2025,2212,18.374264,0.008307,16.014959,0.007240
2,Antoine Semenyo,"Bournemouth,Manchester City",2025,2250,9.511308,0.004227,12.810340,0.005693
3,Dominic Calvert-Lewin,Leeds,2025,1671,10.760022,0.006439,11.817670,0.007072
4,João Pedro,Chelsea,2025,1913,9.348644,0.004887,12.301924,0.006431


In [7]:
# Plotly scatter: xG per minute vs minutes played for attacking players

fig_xg = px.scatter(
    attacking,
    x="time",
    y="xG_per_min",
    color="season",
    hover_name="player_name",
    hover_data={
        "team_title": True,
        "league": True,
        "time": True,
        "xG": True,
        "xG_per_min": ':.4f',
    },
    trendline="ols",
    trendline_color_override="red",
    title="Top-5 leagues attacking players: xG per minute vs minutes played",
    labels={
        "time": "Minutes played",
        "xG_per_min": "xG per minute",
    },
)

fig_xg.show()

In [8]:
# Plotly scatter: xGChain per minute vs minutes played for attacking players

fig_xgchain = px.scatter(
    attacking,
    x="time",
    y="xGChain_per_min",
    color="season",
    hover_name="player_name",
    hover_data={
        "team_title": True,
        "league": True,
        "time": True,
        "xGChain": True,
        "xGChain_per_min": ':.4f',
    },
    trendline="ols",
    trendline_color_override="red",
    title="Top-5 leagues attacking players: xGChain per minute vs minutes played",
    labels={
        "time": "Minutes played",
        "xGChain_per_min": "xGChain per minute",
    },
)

fig_xgchain.show()

In [15]:
# Plotly scatter: xG and xA for attacking players

fig_xgchain = px.scatter(
    attacking,
    x="time",
    y="xA",
    color="season",
    hover_name="player_name",
    hover_data={
        "team_title": True,
        "league": True,
        "time": True,
        "xGChain": True,
        "xGChain_per_min": ':.4f',
    },
    trendline="ols",
    trendline_color_override="red",
    title="Top-5 leagues attacking players: xA per minute vs minutes played",
    labels={
        "time": "Minutes played",
        "xA": "xA per minute",
    },
)

fig_xgchain.show()

In [10]:
# Load team xG and goals for all leagues (for team_xG and team_goals by league)
all_teams_xg = []
for league_code in LEAGUE_CODES:
    league_csv_path = DATA_DIR / str(SEASON_START_YEAR) / f"{league_code}_teams_xg_last_5_seasons.csv"
    print(f"Loading {league_csv_path} ...")
    league_df = pd.read_csv(league_csv_path)[["Team", "xG_2025", "G_2025"]]
    league_df["league"] = league_code
    all_teams_xg.append(league_df)
all_teams_xg = pd.concat(all_teams_xg, ignore_index=True)
# Keep one copy per (Team, league) for merge
all_teams_xg = all_teams_xg.drop_duplicates(subset=["Team", "league"])

Loading /Users/markdavis/Projects/soccer-stats/data/2025/Ligue_1_teams_xg_last_5_seasons.csv ...


In [11]:
all_teams_xg.head()

,Team,xG_2025,G_2025
0,Ajaccio,NaN,NaN
1,Angers,22.199981,22.0
2,Auxerre,25.421974,17.0
3,Bordeaux,NaN,NaN
4,Brest,31.650537,29.0


In [27]:
attacking["goal_scoring_efficiency"] = attacking["xG"] / attacking["goals"]
attacking["goal_creating_efficiency"] = attacking["xA"] / attacking["assists"]
# Add team_xG and team_goals from all leagues (merge on team_title + league)
attacking = attacking.merge(
    all_teams_xg,
    left_on=["team_title", "league"],
    right_on=["Team", "league"],
    how="left",
    suffixes=("", "_team"),
)
attacking["team_xG"] = attacking["xG_2025"]
attacking["team_goals"] = attacking["G_2025"]
attacking = attacking.drop(columns=["Team", "xG_2025", "G_2025"])

attacking["goal_contribution_to_team"] = attacking["goals"] + attacking["assists"] / attacking["team_goals"]
attacking.sort_values(by="goal_contribution_to_team", ascending=False).head(50)

,id,player_name,games,time,goals,xG,assists,xA,shots,key_passes,...,season,league,uefa_coefficient,xG_per_min,xGChain_per_min,goal_scoring_efficiency,goal_creating_efficiency,team_xG,team_goals,goal_contribution_to_team
2080,12413,Joaquín Panichelli,22,1864,12,12.197169,1,2.952475,48,21,...,2025,Ligue_1,79.212,0.006544,0.006845,1.016431,2.952475,36.676141,36.676141,12.027266
2083,13735,Pavel Sulc,20,1238,9,7.207631,3,2.644356,36,14,...,2025,Ligue_1,79.212,0.005822,0.009642,0.800848,0.881452,37.683381,37.683381,9.079611
2086,9656,Ilan Kebbal,21,1824,8,5.306021,4,3.513580,43,45,...,2025,Ligue_1,79.212,0.002909,0.006206,0.663253,0.878395,28.118753,28.118753,8.142254
2085,3697,Odsonne Edouard,18,1177,8,7.977008,3,1.552698,29,11,...,2025,Ligue_1,79.212,0.006777,0.007693,0.997126,0.517566,47.948917,47.948917,8.062567
2088,6846,Sofiane Diop,21,1445,7,5.978489,2,2.527953,42,22,...,2025,Ligue_1,79.212,0.004137,0.006948,0.854070,1.263976,29.604813,29.604813,7.067557
2087,3721,Romain Del Castillo,20,1667,7,7.018019,2,4.870667,23,42,...,2025,Ligue_1,79.212,0.004210,0.004890,1.002574,2.435333,31.650537,31.650537,7.063190
2090,10016,Bradley Barcola,19,1092,7,8.221372,1,2.708938,45,26,...,2025,Ligue_1,79.212,0.007529,0.012688,1.174482,2.708938,47.315415,47.315415,7.021135
2091,10347,Pablo Pagis,19,1143,7,5.567483,0,2.025036,37,18,...,2025,Ligue_1,79.212,0.004871,0.005859,0.795355,inf,31.408306,31.408306,7.000000
2095,5727,Gauthier Hein,20,1603,6,4.977401,4,2.344767,27,31,...,2025,Ligue_1,79.212,0.003105,0.003384,0.829567,0.586192,20.805554,20.805554,6.192256
2097,10815,Lassine Sinayoko,21,1832,6,7.521358,3,3.387954,55,26,...,2025,Ligue_1,79.212,0.004106,0.004324,1.253560,1.129318,25.421974,25.421974,6.118008


In [28]:
attacking["league"].value_counts()

league
La_liga       71
EPL           62
Serie_A       54
Bundesliga    42
Ligue_1       39
Name: count, dtype: int64

In [21]:
attacking["goal_contribution_to_team"].sort_values(ascending=False).head(10)

2080    12.027266
2083     9.079611
2086     8.142254
2085     8.062567
2088     7.067557
2087     7.063190
2090     7.021135
2091     7.000000
2095     6.192256
2097     6.118008
Name: goal_contribution_to_team, dtype: float64